In [3]:
import torch
import numpy as np

tensor= torch.ones(2, 4, 4)
tensor[:,1] = 0
print(tensor)
print(f"Firstrow: {tensor[0]}")
print(f"Firstcolumn: {tensor[:, 0]}")
print(f"Lastcolumn: {tensor[..., -1]}")


tensor([[[1., 1., 1., 1.],
         [0., 0., 0., 0.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [0., 0., 0., 0.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
Firstrow: tensor([[1., 1., 1., 1.],
        [0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Firstcolumn: tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Lastcolumn: tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [4]:
a= torch.arange(8).reshape(2,2,2)
print(f"a:\n{a}\n")
print(f"a[0]:\n{a[0]}\n")
print(f"a[:,0]:\n{a[:,0]}\n")
print(f"a[:,:,0]:\n{a[:,:,0]}\n")
print(f"a[...,0]:\n{a[...,0]}\n")

a:
tensor([[[0, 1],
         [2, 3]],

        [[4, 5],
         [6, 7]]])

a[0]:
tensor([[0, 1],
        [2, 3]])

a[:,0]:
tensor([[0, 1],
        [4, 5]])

a[:,:,0]:
tensor([[0, 2],
        [4, 6]])

a[...,0]:
tensor([[0, 2],
        [4, 6]])



In [ ]:
x= torch.tensor([1.0])
x.item() # 1.0

1.0

In [ ]:
t= torch.ones(5)
print(f"t: {t}")
n= t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


In [ ]:

t= torch.ones(5)
print(f"t: {t}")
n= t.numpy()
print(f"n: {n}")
t+=1
print(f"t: {t}")
print(f"n: {n}")

#tensor([1,1,1,1,1])
#[1,1,1,1,1]


t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]
t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


In [ ]:

n= np.ones(5)
t= torch.from_numpy(n)
n = n +1
print(f"t: {t}")
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


# **Polynomial Regression**

In [5]:
dtype= torch.float
device= torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU

In [24]:
# Create Tensors to hold input and outputs.
import numpy as np
import torch
x = np.linspace(-np.pi , np.pi , 1000)
y  = np.sin(x)
x= torch.from_numpy(x)
y= torch.from_numpy(y)

In [51]:
# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
a = torch.randn(() , device = device , dtype= dtype , requires_grad = True)
b= torch.randn((), device=device, dtype=dtype, requires_grad=True)
c= torch.randn((), device=device, dtype=dtype, requires_grad=True)
d= torch.randn((), device=device, dtype=dtype, requires_grad=True)

print(a)

tensor(1.3258, requires_grad=True)


# no Optimezer , no optim.step() , zero_grad()

In [50]:
learning_rate = 1e-3

def compute_y_pred(x):
  return a + b * x + c * (x ** 2) + d * (x ** 3)

def compute_loss(y_true , y_pred):
  return torch.mean((y_true - y_pred)**2)

for t in range(4000):
  y_pred = compute_y_pred(x)
  loss = compute_loss(y,y_pred)
  if ((t+1) % 500 ==0):
    print("loss: " , loss.item())
  loss.backward()
  with torch.no_grad():
    a -= learning_rate * a.grad
    b -= learning_rate * b.grad
    c -= learning_rate * c.grad
    d -= learning_rate * d.grad
  a.grad = None
  b.grad = None
  c.grad = None
  d.grad = None




loss:  1.726174784414381
loss:  0.6320465317755855
loss:  0.23410606366188128
loss:  0.08884704404924731
loss:  0.03560673905924796
loss:  0.016004376634851013
loss:  0.008750730854286884
loss:  0.006051817718638862


In [52]:
learning_rate = 1e-3
optim = torch.optim.SGD([a,b,c,d], lr=learning_rate)
criterian = torch.nn.MSELoss(reduction = 'mean')

def compute_y_pred(x):
  return a + b * x + c * (x ** 2) + d * (x ** 3)

def compute_loss(y_true , y_pred):
  return criterian(y_true,y_pred)

for t in range(4000):
  y_pred = compute_y_pred(x)
  loss = compute_loss(y,y_pred)
  if ((t+1) % 500 ==0):
    print("loss: " , loss.item())
  optim.zero_grad()
  loss.backward()
  optim.step()





loss:  0.3304691041719984
loss:  0.13825009027858595
loss:  0.05955102718067113
loss:  0.027205287156323366
loss:  0.01386552903114587
loss:  0.008347465480589332
loss:  0.0060588648699371455
loss:  0.005107498224820409


# **with torch.nn.Module**

In [70]:
# 데이터 생성
x = torch.linspace(-np.pi, np.pi, 1000)
y = torch.sin(x)

class Polynomial3(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.a = torch.nn.Parameter(torch.randn(()))
    self.b = torch.nn.Parameter(torch.randn(()))
    self.c = torch.nn.Parameter(torch.randn(()))
    self.d = torch.nn.Parameter(torch.randn(()))

  def forward(self, x):
    return self.a + self.b * x + self.c * (x**2) + self.d * (x**3)

  def string(self ):
    return f'y= {self.a.item()} + {self.b.item()}x + {self.c.item()}x^2 + {self.d.item()}x^3'


In [71]:
model = Polynomial3()
criterion = torch.nn.MSELoss(reduction = 'mean')
optim = torch.optim.SGD(model.parameters(),lr = learning_rate)

for t in range(4000):
    # Forward pass: Compute predicted y by passing x to the model
    y_pred = model(x)
    loss = criterion(y,y_pred)
    if ((t+1) % 500 ==0):
      print("loss: " , loss.item())
    loss.backward()
    optim.step()
    optim.zero_grad()

print(f'Result: {model.string()}')

loss:  0.041482795029878616
loss:  0.019803792238235474
loss:  0.010822015814483166
loss:  0.00709174619987607
loss:  0.005539235658943653
loss:  0.0048919133841991425
loss:  0.004621582571417093
loss:  0.0045085386373102665
Result: y= -0.013309425674378872 + 0.8587110638618469x + 0.0022937029134482145x^2 + -0.09358757734298706x^3


# **Dataset & DataLoader 사용**

In [72]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, xs, ys):
        self.xs= xs
        self.ys= ys

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        x= self.xs[idx]
        y= self.ys[idx]
        return x, y
my_dataset= MyDataset(x, y)
print(f'{len(my_dataset) = }')
print(f'{my_dataset[0] = }')

len(my_dataset) = 1000
my_dataset[0] = (tensor(-3.1416), tensor(8.7423e-08))


In [73]:
from torch.utils.data import Dataset, DataLoader
batch_size= 200
data_loader= DataLoader(my_dataset, batch_size, shuffle=True)
for X, y in data_loader:
    print(f"{X.shape= }")
    print(f"{y.shape= }")
    break

X.shape= torch.Size([200])
y.shape= torch.Size([200])


In [74]:
t= 0
for e in range(800):
    for _x, _y in data_loader:
        # Forward pass: Compute predicted y by passing x to the model
        y_pred  = model(_x)
        # Compute and print loss
        loss= criterion(y_pred, _y)
        if(t+ 1) % 200== 0:
          print(t+ 1, loss.item())
        # Zero gradients, perform a backward pass, and update the weights.
        optim.zero_grad()
        loss.backward()
        optim.step()
    t+= 1

200 0.004089596681296825
200 0.005010654218494892
200 0.004000249784439802
200 0.004342916887253523
200 0.0048135025426745415
400 0.004150086548179388
400 0.004501141142100096
400 0.004264648538082838
400 0.004247000440955162
400 0.005047850776463747
600 0.004202819894999266
600 0.004410977941006422
600 0.004609373398125172
600 0.004444518126547337
600 0.0044851889833807945
800 0.00474928691983223
800 0.004377654287964106
800 0.004708412103354931
800 0.004104624502360821
800 0.004217429552227259
